# Encoding and log transform: who benefits?

Same features as notebook 04 (application + engineered + `bureau`, `previous`, `installments`). We test two preprocessing steps on two model families:

- **LightGBM** (trees): splits on thresholds, so a log transform cannot change its predictions, and native category handling usually matches one-hot. Expect little or no change.
- **Logistic regression** (linear): sensitive to skew, scale and how categories are encoded. Expect a real effect.

Linear models also need missing values filled and features scaled; both happen inside a pipeline so each CV fold learns them from its own training data only.

The first code cell below rebuilds the same feature matrix as notebook 04.

In [1]:
# Setup: rebuild the full feature matrix from notebook 04 (application + three aggregated tables)
import sys
sys.path.append("..")  # so src/ can be imported from the notebooks folder

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from src.features import clean, add_features, skewed_columns, log_transform, one_hot

df = pd.read_csv("../data/raw/HC_application_train.csv")
y = df["TARGET"]
X = add_features(clean(df.drop(columns="TARGET")))
# Join each pre-aggregated table onto the applicants by SK_ID_CURR
for name in ["bureau", "previous", "installments"]:
    X = X.join(pd.read_pickle(f"../data/processed/{name}.pkl"), on="SK_ID_CURR")
# Drop the ID (a key, not a feature) and turn infinities from ratios into NaN
X = X.drop(columns="SK_ID_CURR").replace([np.inf, -np.inf], np.nan)
X.shape  # (applicants, features)

(307511, 518)

**Insight:** The combined feature matrix has 307,511 rows and 518 columns.

## 1. What would the log transform touch?

Before testing anything we check how many columns the log transform would touch, and confirm it actually reduces skew on a few examples.

In [2]:
# Find strongly right-skewed columns and show the effect of log1p on three examples
skewed = skewed_columns(X)
print(f"{len(skewed)} of {X.select_dtypes('number').shape[1]} numeric columns are strongly right-skewed (skew > 2, non-negative)")

check = ["AMT_INCOME_TOTAL", "AMT_CREDIT", "CREDIT_INCOME_RATIO"]
# skew near 0 means symmetric; log1p = log(1 + x) is safe for zeros
pd.DataFrame({"skew before": X[check].skew(), "skew after log1p": np.log1p(X[check]).skew()}).round(2)

262 of 502 numeric columns are strongly right-skewed (skew > 2, non-negative)


,skew before,skew after log1p
AMT_INCOME_TOTAL,391.56,0.17
AMT_CREDIT,1.23,-0.34
CREDIT_INCOME_RATIO,1.84,0.20


**Insight:** 262 of 502 numeric columns are strongly right-skewed, so the log transform would touch about half of them. It works on the examples: `AMT_INCOME_TOTAL` skew falls from 391.56 to 0.17 and `CREDIT_INCOME_RATIO` from 1.84 to 0.20, while `AMT_CREDIT` (1.23) is only mildly skewed to begin with.

## 2. LightGBM: does preprocessing change anything?

We re-score LightGBM with one-hot encoding and log1p to see whether either changes its AUC.

In [3]:
# LightGBM with one-hot encoding and log1p, compared with the notebook 04 result
def cv_lgbm(X_):
    X_ = X_.copy()
    # LightGBM reads text columns natively once they are typed as category
    for c in X_.select_dtypes(include=["object", "string"]):
        X_[c] = X_[c].astype("category")
    model = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, colsample_bytree=0.3,
                               random_state=42, verbose=-1)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)  # same 5 folds as notebook 04
    s = cross_val_score(model, X_, y, cv=cv, scoring="roc_auc")
    return round(s.mean(), 4), round(s.std(), 4), X_.shape[1]

# Reference row copied from notebook 04 (not recomputed here)
lgbm_results = [
    {"variant": "native categories, no log (notebook 04 result)", "AUC mean": 0.7868, "AUC std": 0.0039, "features": 518},
]
Xb = one_hot(log_transform(X, skewed))  # log1p on skewed columns, then one-hot for categories
mean, std, n = cv_lgbm(Xb)
lgbm_results.append({"variant": "one-hot + log1p", "AUC mean": mean, "AUC std": std, "features": n})
pd.DataFrame(lgbm_results)

,variant,AUC mean,AUC std,features
0,"native categories, no log (notebook 04 result)",0.7868,0.0039,518
1,one-hot + log1p,0.7867,0.0044,656


**Insight:** One-hot plus log1p gives 0.7867 against 0.7868 for native categories, a difference of 0.0001, which is noise against a fold std of about 0.004. This matches the expectation for trees: neither step helps, and one-hot only inflates the feature count from 518 to 656.

## 3. Logistic regression: raw vs one-hot vs one-hot + log

Pipeline: median imputation, standardisation, L2 logistic regression. 3-fold CV to keep the run time reasonable.

The code below scores the three variants side by side.

In [4]:
# Logistic regression on three preprocessing variants (3-fold CV)
def cv_logreg(X_):
    X_ = X_.astype("float32")  # smaller memory footprint for 500+ columns
    # Imputer and scaler sit inside the pipeline so each fold fits them on its training part only
    # C=0.05 is strong L2 regularisation, which suits this many correlated features
    model = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                          LogisticRegression(C=0.05, max_iter=300))
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)  # 3 folds to keep run time down
    s = cross_val_score(model, X_, y, cv=cv, scoring="roc_auc")
    return round(s.mean(), 4), round(s.std(), 4), X_.shape[1]

numeric_only = X.select_dtypes("number")  # baseline: ignore the text columns entirely
variants = {
    "numeric columns only, raw": numeric_only,
    "+ one-hot categories": one_hot(X),
    "+ one-hot + log1p of skewed columns": one_hot(log_transform(X, skewed)),
}

lr_results = []
for name, X_ in variants.items():
    mean, std, n = cv_logreg(X_)
    lr_results.append({"variant": name, "AUC mean": mean, "AUC std": std, "features": n})
    print(lr_results[-1])
pd.DataFrame(lr_results)

{'variant': 'numeric columns only, raw', 'AUC mean': np.float64(0.7649), 'AUC std': np.float64(0.0013), 'features': 502}


{'variant': '+ one-hot categories', 'AUC mean': np.float64(0.7716), 'AUC std': np.float64(0.0015), 'features': 656}


{'variant': '+ one-hot + log1p of skewed columns', 'AUC mean': np.float64(0.7737), 'AUC std': np.float64(0.0016), 'features': 656}


,variant,AUC mean,AUC std,features
0,"numeric columns only, raw",0.7649,0.0013,502
1,+ one-hot categories,0.7716,0.0015,656
2,+ one-hot + log1p of skewed columns,0.7737,0.0016,656


**Insight:** For logistic regression, adding one-hot categories raises AUC from 0.7649 to 0.7716 (+0.0067), and adding log1p on top gives 0.7737 (+0.0021). The one-hot gain is several times the 3-fold std of about 0.0015, so it is real; the log gain is small and close to noise level. Caveat: this uses 3 folds while LightGBM uses 5, so the comparison with LightGBM (0.7868) is loose, but linear models trail the trees here by about 0.013 AUC.